In [1]:
from datasets import load_dataset
import json
from pathlib import Path

C:\Users\Bikram\anaconda3\envs\earning_call-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds = load_dataset("kurry/sp500_earnings_transcripts")
print("Total transcripts in dataset:", len(ds["train"]))

Total transcripts in dataset: 33362


In [13]:
COMPANIES = ["AAPL", "TSLA", "JPM", "GS", "WTW", "AON", "AMZN"]

RAW_DIR = Path("raw_transcripts")
RAW_DIR.mkdir(exist_ok=True)

print("Companies:", COMPANIES)
print("Folder created at:", RAW_DIR.resolve())

Companies: ['AAPL', 'TSLA', 'JPM', 'GS', 'WTW', 'AON', 'AMZN']
Folder created at: C:\Users\Bikram\earningsiq\data\raw_transcripts


In [14]:
filtered = ds["train"].filter(lambda row: row["symbol"] in COMPANIES)

print("Total transcripts for 7 companies:", len(filtered))

Filter: 100%|██████████| 33362/33362 [01:09<00:00, 477.04 examples/s]

Total transcripts for 7 companies: 419


In [15]:
for company in COMPANIES:
    company_data = filtered.filter(lambda row: row["symbol"] == company)
    years = sorted(set(company_data["year"]))
    print(company, "- years available:", years[-5:])

Filter: 100%|██████████| 419/419 [00:00<00:00, 452.86 examples/s]


AAPL - years available: [2021, 2022, 2023, 2024, 2025]


Filter: 100%|██████████| 419/419 [00:00<00:00, 567.01 examples/s]


TSLA - years available: [2021, 2022, 2023, 2024, 2025]


Filter: 100%|██████████| 419/419 [00:00<00:00, 468.14 examples/s]


...

WTW - years available: [2021, 2022, 2023, 2024, 2025]


Filter: 100%|██████████| 419/419 [00:00<00:00, 466.78 examples/s]


AON - years available: [2021, 2022, 2023, 2024, 2025]


Filter: 100%|██████████| 419/419 [00:00<00:00, 480.70 examples/s]

AMZN - years available: [2021, 2022, 2023, 2024, 2025]


In [16]:
Taken_yrs = [2023, 2024]

saved_count = 0
skipped_count = 0

for company in COMPANIES:
    company_data = filtered.filter(lambda row: row["symbol"] == company and row["year"] in Taken_yrs)
    print(company, "- found", len(company_data), "transcripts for 2023-2024")
    
    for row in company_data:
        quarter = row["quarter"]
        year = row["year"]
        filename = RAW_DIR / f"{company}_Q{quarter}_{year}.json"
        
        if filename.exists():
            skipped_count = skipped_count + 1
            continue
        
        record = {
            "symbol": row["symbol"],
            "company_name": row["company_name"],
            "quarter": quarter,
            "year": year,
            "date": row["date"],
            "content": row["content"],
            "structured_content": row["structured_content"]
        }
        
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        
        saved_count = saved_count + 1

print()
print("Total saved:", saved_count)
print("Total skipped (already existed):", skipped_count)

Filter: 100%|██████████| 419/419 [00:00<00:00, 870.08 examples/s]


AAPL - found 8 transcripts for 2023-2024


Filter: 100%|██████████| 419/419 [00:00<00:00, 469.43 examples/s]


TSLA - found 8 transcripts for 2023-2024


Filter: 100%|██████████| 419/419 [00:00<00:00, 476.60 examples/s]


...

WTW - found 8 transcripts for 2023-2024


Filter: 100%|██████████| 419/419 [00:00<00:00, 464.76 examples/s]


AON - found 8 transcripts for 2023-2024


Filter: 100%|██████████| 419/419 [00:00<00:00, 445.47 examples/s]

AMZN - found 8 transcripts for 2023-2024

Total saved: 56
Total skipped (already existed): 0


In [17]:
sample_file = RAW_DIR / "AAPL_Q1_2023.json"

with open(sample_file, encoding="utf-8") as f:
    sample = json.load(f)

print("Company:", sample["company_name"])
print("Quarter:", sample["quarter"], "Year:", sample["year"])
print("Word count:", len(sample["content"].split()))
print("Number of speaker turns:", len(sample["structured_content"]))
print()
print("First speaker turn:")
print(sample["structured_content"][0])

Company: Apple Inc.
Quarter: 1 Year: 2023
Word count: 7761
Number of speaker turns: 54

First speaker turn:
{'speaker': 'Operator', 'text': "Good day, everyone, and welcome to the Apple Q1 Fiscal Year 2023 Earnings Conference Call. Today's call is being recorded.  And now at this time, for opening remarks and introductions, I would like to turn the call over to Tejas Gala, Director of Investor Relations and Corporate Finance. Please go ahead."}


In [2]:
NEW_COMPANIES = ["MS", "MET", "MSFT"]
NEW_YEARS = [2021, 2022, 2023, 2024]

OLD_COMPANIES = ["AAPL", "TSLA", "JPM", "GS", "WTW", "AON", "AMZN"]
OLD_YEARS_ALREADY_DONE = [2023, 2024]

print("New companies to add:", NEW_COMPANIES)
print("Years for new companies:", NEW_YEARS)
print()
print("Old companies, extending with these years:", OLD_YEARS_ALREADY_DONE, "-> adding", [2021, 2022])

New companies to add: ['MS', 'MET', 'MSFT']
Years for new companies: [2021, 2022, 2023, 2024]

Old companies, extending with these years: [2023, 2024] -> adding [2021, 2022]


In [5]:
all_symbols_check = set(ds["train"]["symbol"])

for company in NEW_COMPANIES:
    print(company, "present:", company in all_symbols_check)

MS present: True
MET present: True
MSFT present: True


In [6]:
new_companies_data = ds["train"].filter(
    lambda row: row["symbol"] in NEW_COMPANIES and row["year"] in NEW_YEARS
)

print("Total transcripts for 3 new companies (2021-2024):", len(new_companies_data))

for company in NEW_COMPANIES:
    company_subset = new_companies_data.filter(lambda row: row["symbol"] == company)
    years_found = sorted(set(company_subset["year"]))
    print(company, "-", len(company_subset), "transcripts | years:", years_found)

Filter: 100%|██████████| 33362/33362 [01:12<00:00, 459.43 examples/s]


Total transcripts for 3 new companies (2021-2024): 48


Filter: 100%|██████████| 48/48 [00:00<00:00, 410.12 examples/s]


MS - 16 transcripts | years: [2021, 2022, 2023, 2024]


Filter: 100%|██████████| 48/48 [00:00<00:00, 466.76 examples/s]


MET - 16 transcripts | years: [2021, 2022, 2023, 2024]


Filter: 100%|██████████| 48/48 [00:00<00:00, 459.54 examples/s]

MSFT - 16 transcripts | years: [2021, 2022, 2023, 2024]


In [7]:
OLD_NEW_YEARS = [2021, 2022]

old_companies_extra_years = ds["train"].filter(
    lambda row: row["symbol"] in OLD_COMPANIES and row["year"] in OLD_NEW_YEARS
)

print("Total transcripts for 7 old companies (2021-2022 only):", len(old_companies_extra_years))

for company in OLD_COMPANIES:
    company_subset = old_companies_extra_years.filter(lambda row: row["symbol"] == company)
    years_found = sorted(set(company_subset["year"]))
    print(company, "-", len(company_subset), "transcripts | years:", years_found)

Filter: 100%|██████████| 33362/33362 [01:08<00:00, 485.01 examples/s]


Total transcripts for 7 old companies (2021-2022 only): 56


Filter: 100%|██████████| 56/56 [00:00<00:00, 439.39 examples/s]


AAPL - 8 transcripts | years: [2021, 2022]


Filter: 100%|██████████| 56/56 [00:00<00:00, 411.81 examples/s]


...

WTW - 8 transcripts | years: [2021, 2022]


Filter: 100%|██████████| 56/56 [00:00<00:00, 410.30 examples/s]


AON - 8 transcripts | years: [2021, 2022]


Filter: 100%|██████████| 56/56 [00:00<00:00, 388.53 examples/s]


AMZN - 8 transcripts | years: [2021, 2022]


In [9]:
def save_new_transcripts(dataset_subset, label):
    saved = 0
    skipped = 0
    
    for row in dataset_subset:
        symbol = row["symbol"]
        quarter = row["quarter"]
        year = row["year"]
        filename = RAW_DIR / f"{symbol}_Q{quarter}_{year}.json"
        
        if filename.exists():
            skipped = skipped + 1
            continue
        
        record = {
            "symbol": row["symbol"],
            "company_name": row["company_name"],
            "quarter": quarter,
            "year": year,
            "date": row["date"],
            "content": row["content"],
            "structured_content": row["structured_content"]
        }
        
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        
        saved = saved + 1
    
    print(label, "- Saved:", saved, "| Skipped (already existed):", skipped)

RAW_DIR = Path("raw_transcripts")
save_new_transcripts(new_companies_data, "New companies (MS, MET, MSFT)")
save_new_transcripts(old_companies_extra_years, "Old companies, 2021-2022")

New companies (MS, MET, MSFT) - Saved: 48 | Skipped (already existed): 0
Old companies, 2021-2022 - Saved: 56 | Skipped (already existed): 0


In [10]:
all_raw_files = list(RAW_DIR.glob("*.json"))
print("Total transcript files now in raw_transcripts/:", len(all_raw_files))

all_companies_final = OLD_COMPANIES + NEW_COMPANIES
for company in all_companies_final:
    count = len([f for f in all_raw_files if f.name.startswith(company + "_")])
    print(company, "-", count, "files")

Total transcript files now in raw_transcripts/: 160
AAPL - 16 files
TSLA - 16 files
JPM - 16 files
GS - 16 files
WTW - 16 files
AON - 16 files
AMZN - 16 files
MS - 16 files
MET - 16 files
MSFT - 16 files
